In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import curve_fit
from scipy.stats import pearsonr

In [ ]:
def binding_curve(conc, Kd, low, high):
    return (high - low)*(conc/(conc + Kd)) + low

In [ ]:
data = pd.read_csv('ciBoNTA_dilution_ELISA_data.csv')
data

In [ ]:
fit_params = []

colors = plt.get_cmap('tab10')
for i in range(0,len(data.columns)-1,10):
    # plt.figure()
    for ii,col in enumerate(data.columns[i+1:min(i+11,len(data.columns))]):
        # plt.plot(data['concs'],data[col],'o',color=colors(ii))

        params,cov = curve_fit(binding_curve, data['concs'], data[col], p0=[1,0,2])
        # print(col,params)

        # x_fit = np.logspace(-3,2.2,1000)
        # y_fit = binding_curve(x_fit,*params)
        # plt.plot(x_fit,y_fit,'-',color=colors(ii))

        fit_params.append([col]+list(params)+list(np.sqrt(np.diag(cov))))
        
    # plt.semilogx()

fit_params = pd.DataFrame(fit_params,columns=['name','Kd','low','high','Kd_err','low_err','high_err'])

In [ ]:
sns.histplot(fit_params['Kd'],bins=np.logspace(-1.2,1.5,12))
plt.semilogx()

In [ ]:
fit_params.head()

In [ ]:
vhh = pd.read_csv('../large_files/JYH3_factor1_gfold_vhh.csv')
vhh.head()

In [ ]:
fam = pd.read_csv('../large_files/JYH3_factor1_gfold_fam.csv')
fam.head()

In [ ]:
cols = ['VHH','enrich_ciBoNTA-JDA','gfold01_ciBoNTA-JDA','clone_id_1_5_3']

seqs = pd.read_csv('selected_sequences.csv').drop_duplicates('VHH name')
seqs = seqs.merge(vhh[cols],left_on='sequence',right_on='VHH',how='left')
seqs = seqs.merge(fam[cols[1:]],on='clone_id_1_5_3',how='left',suffixes=('_vhh','_fam'))
del seqs['sequence']
seqs.head()

In [ ]:
fit_params = fit_params.merge(seqs,left_on='name',right_on='VHH name',how='left')
fit_params.head()

In [ ]:
print(len(fit_params))
fit_params = fit_params[fit_params['clone_id_1_5_3'].notna()].reset_index()
print(len(fit_params))

In [ ]:
n = 70
for i,row in fit_params.iterrows():
    name = row['name']
    ec50 = row['Kd']
    seq = row['VHH']
    seq = '\\\\'.join([seq[i:i+n] for i in range(0,len(seq),n)])
    print(f'{name} & {ec50:.2f} & '+'\\makecell[l]{'+seq+'} \\\\')
    print('\hline')

In [ ]:
plt.figure(figsize=[13,4.3])
letters = 'ABC'

N_per = 8
for s,start in enumerate(range(0,24,N_per)):
    ax = plt.subplot(1,3,s+1)
    for i in range(start,min(start+N_per,len(fit_params))):
        name = fit_params.loc[i,'name']
        plt.plot(data['concs'],data[name],'o',color=colors(i%N_per),label=name)

        x_fit = np.logspace(-3,2.2,1000)
        y_fit = binding_curve(x_fit,*fit_params.loc[i,['Kd','low','high']].tolist())
        plt.plot(x_fit,y_fit,'-',color=colors(i%N_per))

    plt.semilogx()
    if s == 0:
        plt.xlabel('Nanobody conc (nM)',fontsize=16,loc='left')
        ax.annotate("", xytext=(0.87, -0.14), xy=(3.7, -0.14), xycoords='axes fraction',
                    arrowprops={'width':1.7,'color':'k'})
        plt.ylabel('ELISA Signal',fontsize=16)
    else:
        plt.xlabel('',fontsize=16)
        plt.ylabel('',fontsize=16)
        
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    ax.text(-0.15, 1.05, letters[s], fontsize=32, transform=ax.transAxes)
    plt.legend()

plt.subplots_adjust(wspace=0.35)

plt.savefig('figS19.png',dpi=300,bbox_inches='tight')